# INF-PROF — Data visualization with Python

## Session 3 learning objectives

By the end of this session, you should be able to choose a plot from the
**question you want to answer**, not from a catalogue of chart types.

We will use pandas, Matplotlib and mainly Seaborn.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = Path("../data")
sns.set_theme(style="whitegrid")

df = pd.read_csv(DATA / "diabetes.csv")
df.head()

## 1. First: audit the data

Before plotting, inspect ranges and suspicious values. In this teaching dataset,
zeros in some clinical measurements represent unavailable measurements rather
than plausible values.

In [ ]:
zero_is_missing = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
df[zero_is_missing] = df[zero_is_missing].replace(0, np.nan)

df[zero_is_missing].isna().sum()

## 2. One numerical variable → distribution

Questions:

- Where is the center?
- How spread out are values?
- Are there skewness, tails or unusual observations?

Use a histogram, possibly with a density curve.

In [ ]:
sns.histplot(data=df, x="Glucose", bins=30, kde=True, color="tab:blue")
plt.title("Distribution of glucose")
plt.show()

## 3. Compare distributions between groups

Use color or facets to compare groups, but keep the plot readable.

In [ ]:
sns.histplot(
    data=df,
    x="Glucose",
    hue="Outcome",
    bins=25,
    element="step",
    stat="density",
    common_norm=False,
    palette="deep",
)
plt.title("Glucose distribution by outcome")
plt.show()

## 4. Numerical variable by category → box/violin plot

A box plot summarizes the median, quartiles and potential outliers.
It is often clearer than overlapping histograms for group comparison.

In [ ]:
sns.boxplot(data=df, x="Outcome", y="Glucose", color="tab:blue")
plt.xticks([0, 1], ["No diabetes", "Diabetes"])
plt.title("Glucose by outcome")
plt.show()

## 5. Two numerical variables → scatter plot

A scatter plot explores association between two measurements.
Color can encode a third variable.

In [ ]:
sns.scatterplot(
    data=df,
    x="BMI",
    y="Glucose",
    hue="Outcome",
    alpha=0.65,
    palette="deep",
)
plt.title("BMI and glucose")
plt.show()

### Correlation is not causation

A visible association does not establish a causal relationship.

Also remember:

- correlation can miss non-linear relationships;
- outliers can dominate;
- missing-data mechanisms can distort patterns.

## 6. A categorical variable → counts or proportions

Bar charts are usually easier to compare than pie charts.

In [ ]:
counts = df["Outcome"].value_counts().rename(index={0: "No diabetes", 1: "Diabetes"})
sns.barplot(x=counts.index, y=counts.values, color="tab:blue")
plt.ylabel("Number of observations")
plt.title("Class balance")
plt.show()

## 7. Correlation matrix → heatmap

A heatmap is useful for an overview, but it should not replace looking at
individual scatter plots.

In [ ]:
corr = df.corr(numeric_only=True)

plt.figure(figsize=(9, 7))
sns.heatmap(corr, cmap="vlag", center=0)
plt.title("Correlation matrix")
plt.tight_layout()
plt.show()

## 8. Pair plots: useful, but selective

A pair plot becomes unreadable with too many variables.
Choose a small subset driven by a question.

In [ ]:
subset = ["Glucose", "BMI", "Age", "Outcome"]
sns.pairplot(df[subset].dropna(), hue="Outcome", corner=True, palette="deep")
plt.show()

## 9. Line charts require an ordered dimension

A line between observations implies continuity or order, usually time.

Do **not** use a line plot just because a column is numeric.
Use it when the x-axis has a meaningful sequence.

In [ ]:
visits = pd.read_csv(DATA / "patients.csv")
visits["visit_date"] = pd.to_datetime(visits["visit_date"])
monthly = (
    visits.set_index("visit_date")
          .resample("MS")
          .size()
          .rename("n_visits")
)

monthly.plot(marker="o")
plt.ylabel("Number of visits")
plt.title("Visits over time")
plt.show()

## 10. Make plots interpretable

A useful figure should have:

- informative axis labels and units;
- a short title that states the question;
- a readable legend;
- sensible ordering;
- enough contrast;
- no decorative element that hides the data.

Avoid relying on color alone when the distinction is clinically important.

## 11. From exploration to communication

Exploratory plots can be quick and numerous.

A final figure should answer **one clear question** and remove visual noise.

Ask:

> What should the reader learn from this figure in ten seconds?

## Take-home checklist

Choose the plot from the question:

- one numeric variable → histogram;
- numeric by group → box/violin/histogram;
- two numeric variables → scatter plot;
- category counts → bar chart;
- many pairwise correlations → heatmap;
- time → line chart.

Then interpret the figure in words.